# Bab 11: Klasterisasi, Model Campuran, dan EM

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab11/data_ipm.py": "\"\"\"Indikator pembangunan manusia 514 kabupaten/kota sintetis (ilustrasi).\n\nEmpat kelompok tersembunyi dengan sebaran berbeda: perkotaan, perdesaan maju,\nperdesaan berkembang, dan daerah tertinggal. Angka tidak mewakili daerah nyata.\n\"\"\"\nimport numpy as np\n\nFITUR = [\"UHH\", \"HLS\", \"RLS\", \"pengeluaran\"]   # tahun, tahun, tahun, juta Rp/thn\nKELOMPOK = [\"perkotaan\", \"perdesaan maju\", \"perdesaan berkembang\", \"tertinggal\"]\n\ndef buat_data(seed=0):\n    rng = np.random.default_rng(seed)\n    n = [98, 160, 196, 60]\n    mu = np.array([[73.5, 14.8, 11.0, 16.0],\n                   [71.8, 13.3, 8.8, 11.4],\n                   [69.6, 12.5, 7.3, 8.9],\n                   [65.0, 10.6, 5.2, 6.4]])\n    sd = np.array([[0.9, 0.5, 0.6, 2.6],\n                   [0.8, 0.4, 0.5, 0.9],\n                   [1.0, 0.4, 0.5, 0.7],\n                   [2.0, 1.2, 1.1, 0.8]])\n    rho = [0.5, 0.85, 0.85, 0.8]           # korelasi antarindikator dalam kelompok\n    X, z = [], []\n    for k in range(4):\n        R = np.full((4, 4), rho[k]); np.fill_diagonal(R, 1)\n        S = R * np.outer(sd[k], sd[k])\n        X.append(rng.multivariate_normal(mu[k], S, n[k])); z += [k] * n[k]\n    X, z = np.vstack(X), np.array(z)\n    p = rng.permutation(len(z))\n    return X[p], z[p]\n",
"bab11/dbscan_titik_panas.py": "import numpy as np\nfrom sklearn.cluster import DBSCAN, KMeans\nfrom sklearn.metrics import adjusted_rand_score as ari\n\n# Titik panas (hotspot) satelit sintetis di wilayah 100 km x 100 km (ilustrasi):\n# dua kawasan gambut berbentuk lengkung, satu kawasan bulat, dan titik sporadis.\nrng = np.random.default_rng(7)\nt = rng.uniform(0, np.pi, 260)\nlengkung1 = np.c_[30 + 25 * np.cos(t), 45 + 25 * np.sin(t)] + rng.normal(0, 1.8, (260, 2))\nt = rng.uniform(0, np.pi, 220)\nlengkung2 = np.c_[55 - 25 * np.cos(t), 50 - 25 * np.sin(t)] + rng.normal(0, 1.8, (220, 2))\nbulat = rng.normal([82, 82], 3.5, (120, 2))\nsporadis = rng.uniform(0, 100, (60, 2))                    # pembakaran kecil, derau\nX = np.vstack([lengkung1, lengkung2, bulat, sporadis])\nz = np.repeat([0, 1, 2, -1], [260, 220, 120, 60])\n\ndb = DBSCAN(eps=5.0, min_samples=8).fit(X)                 #<1>\nl = db.labels_\nprint(f\"DBSCAN : {l.max() + 1} klaster, {np.sum(l == -1)} titik derau\")\nprint(\"         ukuran klaster:\", np.bincount(l[l >= 0]))\ninti = np.zeros(len(X), bool); inti[db.core_sample_indices_] = True   #<2>\nprint(f\"         titik inti {inti.sum()}, titik tepi {np.sum(~inti & (l >= 0))}\")\nkm = KMeans(3, n_init=10, random_state=0).fit_predict(X)\nm = z >= 0                                                 # bandingkan pada titik non-derau\nprint(f\"ARI (tanpa titik sporadis): DBSCAN {ari(z[m], l[m]):.2f}, K-means {ari(z[m], km[m]):.2f}\")\nfor eps in [2.0, 3.5, 5.0, 8.0, 12.0]:                     # kepekaan terhadap eps  #<3>\n    l2 = DBSCAN(eps=eps, min_samples=8).fit_predict(X)\n    print(f\"eps = {eps:4.1f} km: {l2.max() + 1:2d} klaster, {np.sum(l2 == -1):3d} derau\")\n",
"bab11/em_gmm.py": "import numpy as np\nfrom scipy.stats import multivariate_normal as mvn\nfrom sklearn.cluster import KMeans\nfrom sklearn.metrics import adjusted_rand_score as ari\nfrom data_ipm import buat_data\n\nX, z = buat_data()\nX2 = X[:, [2, 3]]; X2 = (X2 - X2.mean(0)) / X2.std(0)\n\ndef em(X, mu, iter_maks=500, tol=1e-6):\n    K, D = mu.shape\n    pi, S = np.full(K, 1 / K), np.array([np.eye(D)] * K)\n    riwayat = []\n    for t in range(iter_maks):\n        p = np.column_stack([pi[k] * mvn.pdf(X, mu[k], S[k]) for k in range(K)])\n        riwayat.append(np.log(p.sum(1)).sum())             # log-likelihood  #<1>\n        g = p / p.sum(1, keepdims=True)                    # langkah E  #<2>\n        Nk = g.sum(0)                                      # langkah M  #<3>\n        pi, mu = Nk / len(X), (g.T @ X) / Nk[:, None]\n        for k in range(K):\n            D_ = X - mu[k]\n            S[k] = (g[:, k, None] * D_).T @ D_ / Nk[k] + 1e-6 * np.eye(D)\n        if t > 0 and riwayat[-1] - riwayat[-2] < tol: break\n    return pi, mu, S, g, riwayat\n\nrng = np.random.default_rng(0)\nfor s in range(5):                                         # beberapa inisialisasi  #<4>\n    *_, g, ll = em(X2, X2[rng.choice(len(X2), 4, replace=False)])\n    print(f\"inisialisasi acak {s}: {len(ll):3d} iterasi, log-lik. {ll[-1]:8.2f},\"\n          f\" ARI {ari(z, g.argmax(1)):.2f}\")\nkm = KMeans(4, n_init=10, random_state=0).fit(X2)\npi, mu, S, g, ll = em(X2, km.cluster_centers_)             # mulai dari K-means\nprint(f\"mulai dari K-means : {len(ll):3d} iterasi, log-lik. {ll[-1]:8.2f},\"\n      f\" ARI {ari(z, g.argmax(1)):.2f}\")\nprint(f\"K-means saja       : ARI {ari(z, km.labels_):.2f}\")\nprint(\"log-lik. iterasi 0,1,2,5,10,20:\", \" \".join(f\"{ll[t]:.0f}\" for t in [0,1,2,5,10,20]))\nragu = np.sort(g, 1)[:, -1] < 0.6                          # responsibility terbesar < 0,6\nprint(f\"kabupaten 'ragu' (responsibility maks. < 0,6): {ragu.sum()} dari {len(X)}\")\n",
"bab11/kmeans_dari_nol.py": "import numpy as np\nfrom data_ipm import buat_data\n\nX, z = buat_data()\nX2 = X[:, [2, 3]]                                    # RLS dan pengeluaran\nX2 = (X2 - X2.mean(0)) / X2.std(0)                   # standardisasi  #<1>\n\ndef kmeans(X, K, rng, iter_maks=50):\n    mu = X[rng.choice(len(X), K, replace=False)]     # inisialisasi acak\n    riwayat = []\n    for _ in range(iter_maks):\n        d2 = ((X[:, None, :] - mu[None]) ** 2).sum(-1)\n        r = d2.argmin(1)                             # langkah E: tugaskan  #<2>\n        riwayat.append(d2[np.arange(len(X)), r].sum())\n        mu_baru = np.array([X[r == k].mean(0) for k in range(K)])  # langkah M  #<3>\n        if np.allclose(mu_baru, mu): break\n        mu = mu_baru\n    return mu, r, riwayat\n\nrng = np.random.default_rng(3)\nmu, r, J = kmeans(X2, 4, rng)\nprint(\"J per iterasi:\", \" \".join(f\"{j:.0f}\" for j in J))\nhasil = [kmeans(X2, 4, np.random.default_rng(s))[2][-1] for s in range(20)]  #<4>\nprint(f\"J akhir dari 20 inisialisasi: min {min(hasil):.1f}, maks {max(hasil):.1f}\")\nprint(\"ukuran klaster:\", np.bincount(r))\n",
"bab11/pilih_k.py": "import numpy as np\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.mixture import GaussianMixture\nfrom sklearn.cluster import KMeans\nfrom sklearn.metrics import silhouette_score, adjusted_rand_score as ari\nfrom data_ipm import buat_data\n\nX, z = buat_data()\nA = StandardScaler().fit_transform(X)                      # keempat indikator\nprint(\" K |    BIC   |    AIC   | silhouette K-means\")\nfor K in range(1, 9):\n    g = GaussianMixture(K, covariance_type=\"full\", n_init=5, random_state=0).fit(A)  #<1>\n    s = \"  -\" if K == 1 else f\"{silhouette_score(A, KMeans(K, n_init=10, random_state=0).fit_predict(A)):.3f}\"\n    print(f\"{K:2d} | {g.bic(A):8.1f} | {g.aic(A):8.1f} | {s}\")\n\nfor jenis in [\"full\", \"diag\", \"spherical\"]:                 # bentuk kovarians  #<2>\n    g = GaussianMixture(4, covariance_type=jenis, n_init=5, random_state=0).fit(A)\n    print(f\"GMM {jenis:9s}: BIC {g.bic(A):7.1f}, ARI {ari(z, g.predict(A)):.2f}\")\nkm = KMeans(4, n_init=10, random_state=0).fit(A)\nprint(f\"K-means      : ARI {ari(z, km.labels_):.2f}\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab11')
print('folder kerja:', os.getcwd())

## Listing 11.1: K-means dari nol pada dua indikator IPM

Berkas: `kode/bab11/kmeans_dari_nol.py`

In [ ]:
import numpy as np
from data_ipm import buat_data

X, z = buat_data()
X2 = X[:, [2, 3]]                                    # RLS dan pengeluaran
X2 = (X2 - X2.mean(0)) / X2.std(0)                   # standardisasi  # (1)

def kmeans(X, K, rng, iter_maks=50):
    mu = X[rng.choice(len(X), K, replace=False)]     # inisialisasi acak
    riwayat = []
    for _ in range(iter_maks):
        d2 = ((X[:, None, :] - mu[None]) ** 2).sum(-1)
        r = d2.argmin(1)                             # langkah E: tugaskan  # (2)
        riwayat.append(d2[np.arange(len(X)), r].sum())
        mu_baru = np.array([X[r == k].mean(0) for k in range(K)])  # langkah M  # (3)
        if np.allclose(mu_baru, mu): break
        mu = mu_baru
    return mu, r, riwayat

rng = np.random.default_rng(3)
mu, r, J = kmeans(X2, 4, rng)
print("J per iterasi:", " ".join(f"{j:.0f}" for j in J))
hasil = [kmeans(X2, 4, np.random.default_rng(s))[2][-1] for s in range(20)]  # (4)
print(f"J akhir dari 20 inisialisasi: min {min(hasil):.1f}, maks {max(hasil):.1f}")
print("ukuran klaster:", np.bincount(r))

## Listing 11.2: EM untuk campuran Gaussian, dari nol

Berkas: `kode/bab11/em_gmm.py`

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal as mvn
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score as ari
from data_ipm import buat_data

X, z = buat_data()
X2 = X[:, [2, 3]]; X2 = (X2 - X2.mean(0)) / X2.std(0)

def em(X, mu, iter_maks=500, tol=1e-6):
    K, D = mu.shape
    pi, S = np.full(K, 1 / K), np.array([np.eye(D)] * K)
    riwayat = []
    for t in range(iter_maks):
        p = np.column_stack([pi[k] * mvn.pdf(X, mu[k], S[k]) for k in range(K)])
        riwayat.append(np.log(p.sum(1)).sum())             # log-likelihood  # (1)
        g = p / p.sum(1, keepdims=True)                    # langkah E  # (2)
        Nk = g.sum(0)                                      # langkah M  # (3)
        pi, mu = Nk / len(X), (g.T @ X) / Nk[:, None]
        for k in range(K):
            D_ = X - mu[k]
            S[k] = (g[:, k, None] * D_).T @ D_ / Nk[k] + 1e-6 * np.eye(D)
        if t > 0 and riwayat[-1] - riwayat[-2] < tol: break
    return pi, mu, S, g, riwayat

rng = np.random.default_rng(0)
for s in range(5):                                         # beberapa inisialisasi  # (4)
    *_, g, ll = em(X2, X2[rng.choice(len(X2), 4, replace=False)])
    print(f"inisialisasi acak {s}: {len(ll):3d} iterasi, log-lik. {ll[-1]:8.2f},"
          f" ARI {ari(z, g.argmax(1)):.2f}")
km = KMeans(4, n_init=10, random_state=0).fit(X2)
pi, mu, S, g, ll = em(X2, km.cluster_centers_)             # mulai dari K-means
print(f"mulai dari K-means : {len(ll):3d} iterasi, log-lik. {ll[-1]:8.2f},"
      f" ARI {ari(z, g.argmax(1)):.2f}")
print(f"K-means saja       : ARI {ari(z, km.labels_):.2f}")
print("log-lik. iterasi 0,1,2,5,10,20:", " ".join(f"{ll[t]:.0f}" for t in [0,1,2,5,10,20]))
ragu = np.sort(g, 1)[:, -1] < 0.6                          # responsibility terbesar < 0,6
print(f"kabupaten 'ragu' (responsibility maks. < 0,6): {ragu.sum()} dari {len(X)}")

## Listing 11.3: Memilih $K$ dan bentuk kovarians dengan BIC

Berkas: `kode/bab11/pilih_k.py`

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score as ari
from data_ipm import buat_data

X, z = buat_data()
A = StandardScaler().fit_transform(X)                      # keempat indikator
print(" K |    BIC   |    AIC   | silhouette K-means")
for K in range(1, 9):
    g = GaussianMixture(K, covariance_type="full", n_init=5, random_state=0).fit(A)  # (1)
    s = "  -" if K == 1 else f"{silhouette_score(A, KMeans(K, n_init=10, random_state=0).fit_predict(A)):.3f}"
    print(f"{K:2d} | {g.bic(A):8.1f} | {g.aic(A):8.1f} | {s}")

for jenis in ["full", "diag", "spherical"]:                 # bentuk kovarians  # (2)
    g = GaussianMixture(4, covariance_type=jenis, n_init=5, random_state=0).fit(A)
    print(f"GMM {jenis:9s}: BIC {g.bic(A):7.1f}, ARI {ari(z, g.predict(A)):.2f}")
km = KMeans(4, n_init=10, random_state=0).fit(A)
print(f"K-means      : ARI {ari(z, km.labels_):.2f}")

## Listing 11.4: DBSCAN untuk titik panas sintetis

Berkas: `kode/bab11/dbscan_titik_panas.py`

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score as ari

# Titik panas (hotspot) satelit sintetis di wilayah 100 km x 100 km (ilustrasi):
# dua kawasan gambut berbentuk lengkung, satu kawasan bulat, dan titik sporadis.
rng = np.random.default_rng(7)
t = rng.uniform(0, np.pi, 260)
lengkung1 = np.c_[30 + 25 * np.cos(t), 45 + 25 * np.sin(t)] + rng.normal(0, 1.8, (260, 2))
t = rng.uniform(0, np.pi, 220)
lengkung2 = np.c_[55 - 25 * np.cos(t), 50 - 25 * np.sin(t)] + rng.normal(0, 1.8, (220, 2))
bulat = rng.normal([82, 82], 3.5, (120, 2))
sporadis = rng.uniform(0, 100, (60, 2))                    # pembakaran kecil, derau
X = np.vstack([lengkung1, lengkung2, bulat, sporadis])
z = np.repeat([0, 1, 2, -1], [260, 220, 120, 60])

db = DBSCAN(eps=5.0, min_samples=8).fit(X)                 # (1)
l = db.labels_
print(f"DBSCAN : {l.max() + 1} klaster, {np.sum(l == -1)} titik derau")
print("         ukuran klaster:", np.bincount(l[l >= 0]))
inti = np.zeros(len(X), bool); inti[db.core_sample_indices_] = True   # (2)
print(f"         titik inti {inti.sum()}, titik tepi {np.sum(~inti & (l >= 0))}")
km = KMeans(3, n_init=10, random_state=0).fit_predict(X)
m = z >= 0                                                 # bandingkan pada titik non-derau
print(f"ARI (tanpa titik sporadis): DBSCAN {ari(z[m], l[m]):.2f}, K-means {ari(z[m], km[m]):.2f}")
for eps in [2.0, 3.5, 5.0, 8.0, 12.0]:                     # kepekaan terhadap eps  # (3)
    l2 = DBSCAN(eps=eps, min_samples=8).fit_predict(X)
    print(f"eps = {eps:4.1f} km: {l2.max() + 1:2d} klaster, {np.sum(l2 == -1):3d} derau")